In [ ]:
# onepass: train Qwen3-0.6B and Qwen3-1.7B, then score both on typed-decisions and the held-out human sets.
# Two GPUs: both models at once, one per GPU. One GPU: one after the other.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.55", "peft",
                "git+https://github.com/samadpls/onepass@main"], check=True)
print(subprocess.run(["git", "ls-remote", "https://github.com/samadpls/onepass", "main"], capture_output=True, text=True).stdout)

In [ ]:
import os, pathlib, threading
import torch
from huggingface_hub import hf_hub_download, snapshot_download
n_gpu = torch.cuda.device_count()
assert n_gpu > 0, "This session has no GPU: Kaggle gave a CPU machine."
print(torch.__version__, n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
DATA = str(next(pathlib.Path("/kaggle/input").rglob("train.jsonl")).parent)
TEST = hf_hub_download("LocalLLaMA/typed-decisions", "all/test-00000-of-00001.parquet", repo_type="dataset")
bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
DTYPE = "bf16" if bf16 else "fp16"
if n_gpu >= 2:   # name: (base model, minutes of training, max tokens per batch)
    RUNS = {"qwen3-0.6b": ("Qwen/Qwen3-0.6B", 540, 8192), "qwen3-1.7b": ("Qwen/Qwen3-1.7B", 540, 4096)}
else:
    RUNS = {"qwen3-0.6b": ("Qwen/Qwen3-0.6B", 150, 8192), "qwen3-1.7b": ("Qwen/Qwen3-1.7B", 330, 8192)}
BASES = {name: snapshot_download(repo) for name, (repo, _, _) in RUNS.items()}
print(DATA, TEST, DTYPE, RUNS)

In [ ]:
def stream(name, proc):
    for line in proc.stdout:
        print(f"[{name}] {line}", end="", flush=True)

def launch(gpu, name, minutes, max_tokens):
    code = (f"from onepass.evaluate import run; "
            f"run({BASES[name]!r}, {DATA!r}, '/kaggle/working/{name}', {TEST!r}, "
            f"max_minutes={minutes}, max_tokens={max_tokens}, dtype={DTYPE!r}, log=lambda s: print(s, flush=True))")
    p = subprocess.Popen([sys.executable, "-u", "-c", code], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t = threading.Thread(target=stream, args=(name, p), daemon=True)
    t.start()
    return p, t

procs = []
for i, (name, (_, minutes, max_tokens)) in enumerate(RUNS.items()):
    p, t = launch(i % n_gpu, name, minutes, max_tokens)
    if n_gpu < 2:
        p.wait(); t.join()
    procs.append((name, p, t))
for name, p, t in procs:
    p.wait(); t.join()
    print(name, "exited with", p.returncode, flush=True)

In [ ]:
import json
for name in RUNS:
    path = pathlib.Path(f"/kaggle/working/{name}/results.json")
    if path.exists():
        r = json.loads(path.read_text())
        for k in ("base", "tuned", "tuned_calibrated"):
            print(name, k, {a: round(b, 4) for a, b in r[k].items() if isinstance(b, float)})
    else:
        print(name, "has no results")